# Session 3 Explorer: The Three Failure Modes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/buildLittleWorlds/level-2-course-material/blob/main/session-03/explorer.ipynb)

Apply the three failure modes to a model from your Collection. In class we tested sentiment and zero-shot models on news headlines and found three ways they fail. Now pick a model from YOUR Collection and find out which failure modes hit it hardest.

## Setup

In [ ]:
!pip install -q transformers torch

In [ ]:
from transformers import pipeline
import json

## Load YOUR Model

Pick a text-classification model from your Collection. Replace `MODEL_NAME` with one of these or another from your own research:

**Sentiment:**
- `distilbert-base-uncased-finetuned-sst-2-english` (SST-2 sentiment)
- `cardiffnlp/twitter-roberta-base-sentiment` (tweet sentiment)
- `nlptown/bert-base-multilingual-uncased-sentiment` (multi-language)

**Emotion:**
- `j-hartmann/emotion-english-distilroberta-base` (6 emotions)
- `SamLowe/roberta-base-go_emotions` (27 emotions)

**Any text-classification model:** as long as it exists on Hugging Face Hub

In [ ]:
# Replace this with your model
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

classifier = pipeline("text-classification", model=MODEL_NAME, top_k=3)
print(f"Model loaded: {MODEL_NAME}")

## Test 1: Tone Deafness

**Does your model miss meaning that's there?**

Tone deafness happens when a model reads surface meaning literally but misses the actual intent. Test with:
1. Sarcasm or irony (positive words, negative meaning)
2. Measured language about bad situations (calm tone, negative content)
3. Positive words describing negative outcomes (upbeat words, bad news)

Write 3 examples that match your model's domain. News headlines, tweets, customer reviews, etc.

In [ ]:
tone_deafness_inputs = [
    "This is just wonderful, another delay in my flight.",
    "The stock market declined steadily throughout the month.",
    "Great news: we've shut down three production facilities."
]

print("\n=== TONE DEAFNESS TEST ===")
for text in tone_deafness_inputs:
    result = classifier(text)
    print(f"\nInput: \"{text}\"")
    print(f"Top 3 predictions:")
    for i, pred in enumerate(result[0], 1):
        print(f"  {i}. {pred['label']}: {pred['score']:.1%}")

### Reflection: Tone Deafness

Did your model catch the real meaning, or read the words literally?

- **Bad**: Model consistently chose wrong sentiment (e.g., marked sarcasm as positive)
- **Moderate**: Sometimes wrong, sometimes right
- **Surprisingly Good**: Caught most of the actual intent despite non-literal language

*Your rating: [ ]* (edit after running)

## Test 2: Emotional Flattening

**Does your model oversimplify complex emotions?**

Emotional flattening happens when a model forces genuinely mixed or nuanced feelings into a single category. It's forced to pick one label even when the real answer is "both" or "neither."

Write 3 examples with legitimately mixed or ambiguous feelings:
1. Bittersweet (happy and sad together)
2. Anxious hope (uncertain but optimistic)
3. Conflicted (wanting two incompatible things)

In [ ]:
flattening_inputs = [
    "I'm excited to start my new job, but I'll miss my old team.",
    "The treatment works sometimes, but nobody knows why or if it'll help me.",
    "It's our last game together, and we just won the championship."
]

print("\n=== EMOTIONAL FLATTENING TEST ===")
for text in flattening_inputs:
    result = classifier(text)
    print(f"\nInput: \"{text}\"")
    print(f"Top 3 predictions:")
    for i, pred in enumerate(result[0], 1):
        print(f"  {i}. {pred['label']}: {pred['score']:.1%}")

### Reflection: Emotional Flattening

Look at the confidence spread. Does the model express uncertainty about mixed emotions?

- **Severe**: Top label is 95%+ confident; model acts like the answer is obvious
- **Moderate**: Top label 70-90%; some confidence but not overwhelming
- **Minimal**: Top label under 70% or spread across multiple labels; model at least uncertain

*Your rating: [ ]* (edit after running)

## Test 3: Anthropomorphic Projection

**Does your model invent emotions that aren't there?**

Anthropomorphic projection happens when a model assigns emotions to factual, emotionless descriptions. A confident emotional prediction on text with zero human feeling is a false positive.

Write 3 emotionless examples:
1. Weather report
2. Technical description or market data
3. Factual statement about animal behavior or geology

In [ ]:
projection_inputs = [
    "High of 72 degrees with 30% chance of afternoon showers.",
    "The Federal Reserve raised interest rates by 0.25 percent.",
    "Penguins breed in colonies containing thousands of birds."
]

print("\n=== ANTHROPOMORPHIC PROJECTION TEST ===")
for text in projection_inputs:
    result = classifier(text)
    print(f"\nInput: \"{text}\"")
    print(f"Top 3 predictions:")
    for i, pred in enumerate(result[0], 1):
        print(f"  {i}. {pred['label']}: {pred['score']:.1%}")

### Reflection: Anthropomorphic Projection

Did the model assign emotions to emotionless text? A confident answer on genuinely neutral input is a problem.

- **Heavy**: Model confidently assigns emotion to all or most emotionless inputs
- **Some**: Model assigns emotion to some emotionless inputs
- **Almost None**: Model mostly treats emotionless input as neutral or low-confidence

*Your rating: [ ]* (edit after running)

## Bonus: Zero-Shot Comparison

Now test the SAME inputs with a zero-shot classifier and custom categories. Can better labels reduce the failure modes?

In [ ]:
zs_model = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Pick one input from each test
zs_tests = [
    ("Tone Deafness", tone_deafness_inputs[0]),
    ("Flattening", flattening_inputs[0]),
    ("Projection", projection_inputs[0]),
]

# Define custom categories
my_categories = ["positive", "negative", "neutral", "mixed"]

print("\n=== ZERO-SHOT COMPARISON ===")
for label, text in zs_tests:
    result = zs_model(text, my_categories)
    scores = [f"{l} ({s:.0%})" for l, s in zip(result['labels'], result['scores'])]
    print(f"\n{label}:")
    print(f'  Input: \"{text}\"')
    print(f"  → {', '.join(scores)}")

### Reflection: Zero-Shot vs. Fine-tuned

Did custom categories help? Or did the model make the same kinds of mistakes with better labels?

- Did zero-shot catch tone deafness better?
- Did it express more uncertainty on mixed emotions?
- Did it avoid projecting emotions onto neutral text?

Write your observations here.

## Failure Profile

| Failure Mode | Rating | Example |
|---|---|---|
| **Tone Deafness** | [ ] | |
| **Emotional Flattening** | [ ] | |
| **Anthropomorphic Projection** | [ ] | |

**Overall profile:** This model is good at _____ but bad at _____.

## The Bigger Picture

Classification is real and useful. Sentiment analysis, emotion detection, and content moderation all rely on it. But the three failure modes are **fundamental limits**. They come from the structure of classification itself:

- **One input, one label.** The model must pick. It can't say "both" (emotional flattening) or "neither" (projection).
- **Words are all it has.** Surface meaning is all the data. It can't know you're being sarcastic unless sarcasm patterns exist in training data (tone deafness).
- **Confidence is built-in.** The model is trained to be confident. It has to assign something. It's not trained to say "this text has no emotion."

These limits aren't bugs. They're features of the classification paradigm itself.

**Bridge to Session 4:** What if we stopped sorting into buckets? What if instead of asking "is this positive or negative?", we asked "what is this text *about*?" or "what feelings *coexist* here?" That's where we go next.

## Research Journal: Week 3 Entry

**Date:** [today]

**Model tested:** [MODEL_NAME]

**Key findings:**
- Tone deafness: [which inputs failed and why]
- Emotional flattening: [examples of forced choices]
- Anthropomorphic projection: [examples of false emotion assignments]

**Most surprising result:**

**Questions for next session:**

**Connection to Session 2:** [How does this model compare to what we tested in class?]

**Connection to Session 4:** [What new approach might avoid these failure modes?]

## Collection Tasting Note

Add this model to your Hugging Face Collection with a tasting note:

**Model:** [MODEL_NAME]

**Collection:** [Your Collection name]

**Date added:** [today]

**Your rating:** ⭐⭐⭐⭐⭐

**Tasting notes:**
- Best at: [what this model does well]
- Blind spots: [the failure modes you found]
- Use case: [when you'd actually use this]
- Compared to: [other models in your Collection]

**Example:**
> This sentiment model is reliable on straightforward opinions but completely misses sarcasm and complex emotions. Good baseline for product reviews, weak for social media. Stronger than my previous choice on emotional nuance but still forces binary classification.